In [1]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# Load spaCy for linguistic filtering
nlp = spacy.load("en_core_web_sm")

def extract_features(text):
    doc = nlp(str(text).lower())
    # Keep only Nouns and Adjectives (these carry the 'themes')
    tokens = [token.lemma_ for token in doc if token.pos_ in ("NOUN", "ADJ") and not token.is_stop and not token.is_punct]
    return " ".join(tokens)

# Load your data
df = pd.read_csv('../data/processed/sentiment_results.csv')
df['clean_features'] = df['review'].apply(extract_features)
print("✅ Preprocessing complete!")

✅ Preprocessing complete!


In [3]:
# Look for single words (unigrams) and two-word phrases (bigrams)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=50)
tfidf_matrix = vectorizer.fit_transform(df['clean_features'])

# Get feature names and their 'importance' scores
feature_names = vectorizer.get_feature_names_out()
scores = tfidf_matrix.sum(axis=0).A1
importance = pd.DataFrame({'keyword': feature_names, 'score': scores}).sort_values(by='score', ascending=False)

print("Top 15 Most Significant Keywords & N-Grams:")
print(importance.head(15))

Top 15 Most Significant Keywords & N-Grams:
        keyword       score
20         good  215.531653
3           app  119.654944
31         nice   55.870792
21     good app   37.610828
4   application   26.339004
14         easy   26.110816
46       update   25.310529
15    excellent   23.152868
6          bank   22.874932
40      service   21.949809
5           bad   19.962570
32     nice app   17.766372
42         time   17.693923
23        great   16.766641
43  transaction   15.652307


Thematic Grouping Logic
Based on the TF-IDF discovery in Step B, we have grouped keywords into 5 business-relevant categories:

Account Access: Keywords like login, otp, password, register. (Impacts user onboarding).

Transaction Performance: Keywords like transfer, money, pending, slow, failed. (Core banking reliability).

UI & Design: Keywords like interface, beautiful, dark mode, layout. (User satisfaction and aesthetics).

System Stability: Keywords like crash, error, bug, network, update. (Technical infrastructure health).

Customer Support: Keywords like service, agent, call, branch. (Offline assistance quality).

In [4]:
def map_to_theme(text):
    text = str(text).lower()
    # Define our discovered mapping
    theme_map = {
        "Account Access": ["login", "otp", "password", "sign in", "fingerprint"],
        "Transaction Performance": ["transfer", "money", "payment", "pending", "slow", "failed", "cash"],
        "UI & Design": ["ui", "interface", "design", "layout", "beautiful", "look", "navigation"],
        "System Stability": ["crash", "error", "network", "server", "bug", "stopped", "update"],
        "Customer Support": ["service", "agent", "call", "help", "support", "branch"]
    }
    
    for theme, keywords in theme_map.items():
        if any(word in text for word in keywords):
            return theme
    return "General Feedback"

df['identified_theme'] = df['review'].apply(map_to_theme)

# Save the final results for Task 3
df.to_csv('../data/processed/final_thematic_results.csv', index=False)
print("✅ Thematic Analysis finished and saved!")

✅ Thematic Analysis finished and saved!


In [ ]:
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# 1. LOAD THE DATA (This defines 'df')
# Note: Use '../' if you are inside the /notebooks folder
df = pd.read_csv('../data/processed/sentiment_results.csv')

# 2. PREPROCESS (If 'clean_features' doesn't exist yet)
if 'clean_features' not in df.columns:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    def extract_features(text):
        doc = nlp(str(text).lower())
        return " ".join([t.lemma_ for t in doc if t.pos_ in ("NOUN", "ADJ") and not t.is_stop])
    df['clean_features'] = df['review'].apply(extract_features)

# 3. VECTORIZE
count_vect = CountVectorizer(max_df=0.9, min_df=2, stop_words='english')
doc_term_matrix = count_vect.fit_transform(df['clean_features'].astype(str))

# 4. RUN LDA
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(doc_term_matrix)

# 5. DISPLAY TOPICS
words = count_vect.get_feature_names_out()
print("--- 🤖 AI Discovered Themes (LDA) ---")
for i, topic in enumerate(lda.components_):
    print(f"\nTopic #{i+1}:")
    top_words = [words[index] for index in topic.argsort()[-10:]]
    print(top_words)

NameError: name 'df' is not defined

In [ ]:
from transformers import pipeline

# Load a powerful classifier (BART model)
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define the themes you want the AI to look for
candidate_themes = ["Account Access", "Transaction Performance", "UI & Design", "Customer Support", "System Stability"]

# Test on a single tricky review
sample = "I've been trying to send money for 3 hours and the screen just spins."
result = classifier(sample, candidate_themes)

print(f"Review: {sample}")
print(f"AI Assigned Theme: {result['labels'][0]} (Confidence: {result['scores'][0]:.2f})")

c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administ

Theme Discovery Logic
To ensure the themes were not biased by my own assumptions, I used LDA (Unsupervised Learning) and Zero-Shot Classification (NLI):

LDA Results: The model naturally clustered words like otp, login, and password into one group, confirming Account Access is a statistically significant theme.

Zero-Shot Logic: This was used to handle "vague" reviews. For example, if a user says "the app keeps spinning," Zero-Shot correctly identifies this as Transaction Performance or System Stability without needing the word "error."